MATH2991 Assignment 1
Yunshen Ji  
z5603229  
Instruments: ASML and ASML_DUAL  
Final Test Duration: 1 hour  
Date of Final Test:[17.7.2026]

1.AIM

The main objective of the algorithm is to profit from both active and passive trading of ASML and ASML_DUAL, which are the exact same stock traded on different exchanges. Passive trading in this algorithm refers to market making, which involves providing liquidity on the Optibook exchange while generating a stable and positive P&L.

Because buyers and sellers rarely arrive at the market in the exact same millisecond with perfectly matching trading volumes, market makers step in to bridge this liquidity gap. By continuously providing bids and asks, market makers absorb short-term supply and demand shocks, narrow the bid-ask spread, and reduce overall market volatility.

My Passive Trading strategy continuously provides two-sided quotes around a calculated fair value, offering immediate liquidity to other participants. Simultaneously, my code employs Active Trading to actively correct the price dislocations between these two dual-listed stocks, aggressively taking orders when the price spread widens beyond our Stellage, thereby generating arbitrage returns.

2.OUTLINE OF STRATEGY

Our algorithm combines “Stellage,” “Crossed Trading,” and “Market Making,” while incorporating a risk-balancing mechanism as a safeguard. The construction of the overall strategy follows a forward logic from data observation to algorithm implementation:

1. Independent Data Collection Script
   Before formally constructing the trading algorithm, I wrote and ran a dedicated order book data collection script. The core purpose of this script was to record real-time order book data between ASML and ASML_DUAL across the two markets at a high frequency. By analysing the collected historical spread data, I obtained information that helped me determine key values, such as S_MAX, and quantify the boundaries of Stellage.

2. Interpolated Stellage
   Based on the data collected by the preliminary script, I established a continuous interpolated target table: when the spread is small, the position changes very little; as the spread becomes larger, the position becomes heavier; and when the spread approaches its historical extreme, the position reaches its maximum. stellage_targets(S) uses interpolation to achieve a smooth transition, avoiding the repeated opening and closing of positions and the idle use of capital caused by hard-coded thresholds.

3. Active Execution & Legging Risk Mitigation
   The algorithm continuously calculates the unified signal S_mid. When the current position P exceeds the target boundary, the algorithm attempts to execute paired trades across the spread using IOC orders. The core constraint is MAX_STELLAGE_CROSS_COST: a trade is triggered only when the cost of crossing the bid–ask spread can mathematically be covered by the expected profit from mean reversion. In addition, when the order book becomes temporarily crossed, the algorithm also checks for pure arbitrage opportunities using CROSSED_MIN_EDGE, in order to capture risk-free profits.

4. The Spill Balancer
   Although spread trading is inherently hedged, incomplete fills can expose us to net directional risk (Spill = Position A + Position B). To solve this problem, I designed a dedicated function, position_balancer. If the absolute exposure exceeds our tolerance level, SPILL_TOLERANCE (10 lots), the balancer automatically identifies the most favourable single-leg buying or selling price currently available in the market and triggers an IOC order to smooth out the directional risk. This ensures that, regardless of how disorderly market trading becomes, the algorithm can remain Delta-neutral.

5. Passive Market Making
   On the passive side, the algorithm does not observe each stock in isolation. It calculates the common_fair value by averaging the mid-prices of ASML and ASML_DUAL. This prevents the program from quoting blindly when the mid-price of a single instrument temporarily becomes inaccurate. To prevent adverse selection and avoid reaching the 100-lot position limit, I introduced the MAX_INVENTORY_SKEW mechanism. As inventory accumulates, the algorithm dynamically shifts its quotes. For example, if the algorithm is long, it lowers both its bid and ask prices, thereby passively encouraging market counterparties to help reduce the risky position.

Future Improvements:
If more time were available, I would modularise and directly embed the preliminary independent data collection script into the main trading program. I would use a rolling window based on recent volatility to dynamically adjust S_MAX, creating an automated closed loop from data collection to dynamic signal tuning, rather than relying on the current static constants.


3.FINAL RESULTS

The performance of the algorithm in one hour
Key Metrics:
    Run Duration: 3600 seconds (60.0 minutes)
    Initial P&L: $4450.40
    Final P&L: $4476.10
    Total Profit Generated: $25.70
    Total Volume Traded / Turnover: 4319 lots

Position and Risk Summary:
    ASML final position: +1
    ASML_DUAL final position: +1
    Final Paired Spread: +0
    Final Spill: +2

Execution Breakdown:
    The total number of orders sent: 22745 (Active IOC: 925 | Passive Limit: 21820)
    Stellage Trigger : 366 times (2527 lots)
    Crossed Arbitrage Trigger: 77 times (1742 lots)
    Balance Trigger: 5 times (50 lots)
    Rate-limit blocks: 0 times

During the testing period, the algorithm performed steadily and generated a total profit of $25.70. The hybrid strategy produced significant results: more than 21,000 passive limit orders continuously provided liquidity and captured the bid–ask spread from normal market fluctuations, while the Stellage and Crossed arbitrage engines actively captured substantial mispricings. The order submission success rate remained between 83% and 95%, effectively avoiding unnecessary slippage.

The Spill Balancer performed fully as expected. After one hour of high-intensity two-sided trading, the final net exposure was only +2 lots. The balancer silently executed 5 trades in the background, totalling 50 lots, continuously absorbing the directional risk caused by asynchronous fills. There were zero rate-limit blocks throughout the entire period, demonstrating that the internal order-throttling mechanism achieved an optimal balance between rapid order submission and exchange rules.

4.EVOLUTION OF ALGORITHM

The algorithm underwent multiple rounds of restructuring and only reached its final robust state after resolving a series of logical flaws that caused one-sided losses. 

The following are the three core stages of its evolution:
    Stage 1: Overcoming “Self-Trading” and “Position Limit Violations”
        The early version only read the existing positions and did not account for outstanding orders, causing the algorithm to frequently exceed the hard position limits.
        Improvement: The get_position_bounds function was introduced to combine outstanding buy and sell orders into a calculation of the “worst-case position.” At the same time, cancel_all_quotes() was forcibly executed before every active order to remove stale passive orders, effectively avoiding self-trading and exchange rejections.

    Stage 2: Fixing the Fatal One-Sided “Zero Available Volume” Bug
        During the intermediate testing stage, the algorithm remained biased towards one-sided positions for long periods and ignored excellent two-way arbitrage opportunities. Investigation revealed a sign error in the calculation of the maximum available selling volume: the original logic, max(0, worst_B - MAX_POS), returned 0 when the position was zero, causing the “Buy A + Sell B” direction to be completely and physically locked.
        Improvement: It was corrected to max(0, worst_B + MAX_POS), ensuring that the position would not fall below the lower limit after selling and fully restoring the algorithm’s two-way arbitrage capability.

    Stage 3: Resolving Signal Conflicts Caused by the “Empty Intersection” of Stellage Target Ranges
        An earlier version attempted to separately calculate the target ranges for the buying spread and selling spread and then simply take their intersection, while overlooking the mathematical possibility that the lower bound could be greater than the upper bound (L_final > U_final). This could cause the algorithm to simultaneously trigger contradictory signals indicating that it should both buy and sell, resulting in the program becoming stuck.
        Improvement:The final version restructured the signal logic and extracted a unified mid-price spread signal:       S_mid = (mid_A - mid_B) / S_MAX 
        A single signal is used to map to a unique target position range, eliminating the logical conflict at its source.


In [ ]:
from optibook.synchronous_client import Exchange
import time
import logging
import datetime

logger = logging.getLogger('client')
logger.setLevel('ERROR')

print('Setup was successful.')

# Parameters

RUN_DURATION = 60*60

LOOP_SLEEP = 0.3

stock_A = 'ASML'
stock_B = 'ASML_DUAL'

# position limit
POSITION_LIMIT = 100
PAIR_POSITION_LIMIT = 100
MAX_PAIR_VOLUME = 30
MAX_BALANCE_VOLUME = 10
SPILL_TOLERANCE = 10

# stellage table
S_MAX = 2.0

# the maximum cost for portfolio rebalancing  crossing cost for stellage
MAX_STELLAGE_CROSS_COST = 0.15

# the minimum value at which profits need in a crossed trade.
CROSSED_MIN_EDGE = 0.075

# market making half price difference
PASSIVE_HALF_SPREAD = 0.75
# number of orders placed on each side
PASSIVE_VOLUME = 5
# maximum position skew
MAX_INVENTORY_SKEW = 0.30

# time for delete orders
CANCEL_WAIT = 0.05


# Statistics
start_time = None
initial_pnl = 0.0

total_orders_sent = 0
active_orders_sent = 0
passive_orders_sent = 0
rate_limit_blocks = 0

stellage_attempts = 0
stellage_trades = 0       # successful stellage executions
stellage_lots = 0

crossed_attempts = 0
crossed_trades = 0        # successful crossed executions
crossed_lots = 0

balance_attempts = 0
balance_trades = 0        # successful balance executions
balance_lots = 0

order_counter = 0
delete_counter = 0
counter_second = -1


# Stellage target table

stellage = {
    # S value: (first target boundary, second target boundary)
    -1.00: (1.00, 1.00),
    -0.70: (0.75, 0.90),
    -0.40: (0.40, 0.55),
    -0.20: (0.10, 0.20),
    -0.10: (0.00, 0.05),
     0.10: (0.00, -0.05),
     0.20: (-0.10, -0.20),
     0.40: (-0.40, -0.55),
     0.70: (-0.75, -0.90),
     1.00: (-1.00, -1.00),
}


# Rate-limit
# control the limit of orders and cancels. orders is less than 15, cancel is less than 10
def check_limit(kind='order'):
    global order_counter, delete_counter, counter_second, rate_limit_blocks

    second_now = int(time.time())
    if second_now != counter_second:
        counter_second = second_now
        order_counter = 0
        delete_counter = 0

    if kind == 'order':
        if order_counter >= 15:
            rate_limit_blocks += 1
            return False
        order_counter += 1
    else:
        if delete_counter >= 10:
            rate_limit_blocks += 1
            return False
        delete_counter += 1

    return True


# Control position
# calculate the worst case position
def get_position_bounds(instrument):
    position = exchange.get_positions()[instrument]
    outstanding = exchange.get_outstanding_orders(instrument)

    pending_bids = 0
    pending_asks = 0

    for order in outstanding.values():
        if order.side == 'bid':
            pending_bids += order.volume
        elif order.side == 'ask':
            pending_asks += order.volume

    most_short = position - pending_asks
    most_long = position + pending_bids
    return most_short, most_long

# the volume we can buy under possion limit
def available_buy_volume(instrument):
    _, most_long = get_position_bounds(instrument)
    return max(0, int(POSITION_LIMIT - most_long))

#  the volume we can sell under possion limit
def available_sell_volume(instrument):
    most_short, _ = get_position_bounds(instrument)
    return max(0, int(most_short + POSITION_LIMIT))

# calculate spread position. positive = long A / short B. negative = short A / long B
def calculate_spread_position(pos_A, pos_B):
    if pos_A > 0 and pos_B < 0:
        return min(pos_A, -pos_B)
    if pos_A < 0 and pos_B > 0:
        return -min(-pos_A, pos_B)
    return 0

# calculate spill, Position A + Position B. should near to 0
def calculate_spill(pos_A, pos_B):
    return pos_A + pos_B


# Order maker
# main order function. and check quantity and speed limits for order
def send_order(instrument, price, volume, side, order_type):
    global total_orders_sent, active_orders_sent, passive_orders_sent

    volume = int(volume)
    if volume <= 0:
        return 0

    if side == 'bid':
        volume = min(volume, available_buy_volume(instrument))
    elif side == 'ask':
        volume = min(volume, available_sell_volume(instrument))
    else:
        print('Unknown side:', side)
        return 0

    if volume <= 0:
        print(f'No position capacity for {side} {instrument}.')
        return 0

    if not check_limit('order'):
        print('Order rate limit reached; order skipped.')
        return 0

    try:
        exchange.insert_order(
            instrument,
            price=price,
            volume=volume,
            side=side,
            order_type=order_type,
        )
    except Exception as error:
        print(f'Order rejected: {instrument} {side} {volume} @ {price}: {error}')
        return 0

    total_orders_sent += 1
    if order_type == 'ioc':
        active_orders_sent += 1
    elif order_type == 'limit':
        passive_orders_sent += 1

    return volume

# cancel the unexecuted order and wait some time to make sure successful
def cancel_all_quotes():
    for instrument in (stock_A, stock_B):
        if check_limit('delete'):
            try:
                exchange.delete_orders(instrument)
            except Exception as error:
                print(f'Could not delete orders in {instrument}: {error}')

    time.sleep(CANCEL_WAIT)

    outstanding_A = exchange.get_outstanding_orders(stock_A)
    outstanding_B = exchange.get_outstanding_orders(stock_B)

    if outstanding_A or outstanding_B:
        print('Old quotes are still outstanding; skipping active trade this loop.')
        return False

    return True

# calculate lots traded
def filled_volume(before_position, after_position, side):
    if side == 'bid':
        return max(0, after_position - before_position)
    return max(0, before_position - after_position)

# buy A and sell B simultaneously, or opposite.
# resolve the pending orders on the thinner side firstly. then based on the trading volume on that side, send orders to the other side to offset them.
def execute_pair(direction, requested_volume, strategy_name):
    order_book_A = exchange.get_last_price_book(stock_A)
    order_book_B = exchange.get_last_price_book(stock_B)

    if (not order_book_A.bids or not order_book_A.asks or not order_book_B.bids or not order_book_B.asks):
        return 0

    if direction == 'buy_spread':
        # Buy A, sell B.
        trade_A = {
            'instrument': stock_A,
            'side': 'bid',
            'price': order_book_A.asks[0].price,
            'top_volume': order_book_A.asks[0].volume,
        }
        trade_B = {
            'instrument': stock_B,
            'side': 'ask',
            'price': order_book_B.bids[0].price,
            'top_volume': order_book_B.bids[0].volume,
        }
        capacity_A = available_buy_volume(stock_A)
        capacity_B = available_sell_volume(stock_B)
    elif direction == 'sell_spread':
        # Sell A, buy B.
        trade_A = {
            'instrument': stock_A,
            'side': 'ask',
            'price': order_book_A.bids[0].price,
            'top_volume': order_book_A.bids[0].volume,
        }
        trade_B = {
            'instrument': stock_B,
            'side': 'bid',
            'price': order_book_B.asks[0].price,
            'top_volume': order_book_B.asks[0].volume,
        }
        capacity_A = available_sell_volume(stock_A)
        capacity_B = available_buy_volume(stock_B)
    else:
        return 0


    current_positions = exchange.get_positions()
    current_spread = calculate_spread_position(current_positions[stock_A], current_positions[stock_B])

    
    if direction == 'buy_spread':
        pair_capacity = max(0, PAIR_POSITION_LIMIT - current_spread)
    else:
        pair_capacity = max(0, PAIR_POSITION_LIMIT + current_spread)

    volume = min(
        int(requested_volume),
        int(trade_A['top_volume']),
        int(trade_B['top_volume']),
        int(capacity_A),
        int(capacity_B),
        int(pair_capacity),
    )

    if volume <= 0:
        print(f'{strategy_name}: no safe paired volume available.')
        return 0

    if trade_A['top_volume'] <= trade_B['top_volume']:
        first_leg, second_leg = trade_A, trade_B
    else:
        first_leg, second_leg = trade_B, trade_A

    positions_before = exchange.get_positions()

    submitted_first = send_order(first_leg['instrument'], first_leg['price'], volume, first_leg['side'], 'ioc')
    if submitted_first <= 0:
        return 0

    positions_after_first = exchange.get_positions()
    first_fill = filled_volume(positions_before[first_leg['instrument']], positions_after_first[first_leg['instrument']], first_leg['side'])
    first_fill = min(first_fill, submitted_first)

    if first_fill <= 0:
        print(f'{strategy_name}: first leg did not fill.')
        return 0

    # Refresh the book before sending the hedge leg.
    refreshed_book = exchange.get_last_price_book(second_leg['instrument'])
    if second_leg['side'] == 'bid':
        if not refreshed_book.asks:
            return 0
        second_price = refreshed_book.asks[0].price
        second_top_volume = refreshed_book.asks[0].volume
    else:
        if not refreshed_book.bids:
            return 0
        second_price = refreshed_book.bids[0].price
        second_top_volume = refreshed_book.bids[0].volume

    second_request = min(first_fill, int(second_top_volume))
    submitted_second = send_order( second_leg['instrument'], second_price, second_request, second_leg['side'], 'ioc'
    )

    positions_after_second = exchange.get_positions()
    second_fill = filled_volume(positions_after_first[second_leg['instrument']], positions_after_second[second_leg['instrument']], second_leg['side'])
    second_fill = min(second_fill, submitted_second)

    paired_fill = min(first_fill, second_fill)
    unmatched = first_fill - second_fill

    if unmatched > 0:
        print(
            f'{strategy_name}: WARNING - {unmatched} lots remain unmatched; '
            f'position balancer will reduce the spill.'
        )

    print(
        f'{strategy_name}: requested={volume}, first_fill={first_fill}, '
        f'second_fill={second_fill}, paired={paired_fill}'
    )
    return paired_fill


# Stellage target interpolation
# based on the current price difference (S value), the upper and lower boundaries are calculated by interpolation in a preset table.
def stellage_targets(S):
    S = max(-1.0, min(1.0, S))
    levels = sorted(stellage.keys())

    if S in stellage:
        return stellage[S]

    for index in range(len(levels) - 1):
        low_S = levels[index]
        high_S = levels[index + 1]

        if low_S <= S <= high_S:
            weight = (S - low_S) / (high_S - low_S)
            low_first, low_second = stellage[low_S]
            high_first, high_second = stellage[high_S]

            return (low_first + weight * (high_first - low_first), low_second + weight * (high_second - low_second))

    return stellage[levels[-1]]


# Active strategy
# active trading
# reads the latest order book for both stocks and calculates the midpoint spread signal (S_mid). 1. Stellar 2. Crossed
def trade_loop():
    global stellage_attempts, stellage_trades, stellage_lots
    global crossed_attempts, crossed_trades, crossed_lots

    positions = exchange.get_positions()
    order_book_A = exchange.get_last_price_book(stock_A)
    order_book_B = exchange.get_last_price_book(stock_B)

    if (not order_book_A.bids or not order_book_A.asks or
            not order_book_B.bids or not order_book_B.asks):
        print('Order book is empty; active strategy skipped.')
        return False

    best_bid_A = order_book_A.bids[0].price
    best_ask_A = order_book_A.asks[0].price
    best_bid_B = order_book_B.bids[0].price
    best_ask_B = order_book_B.asks[0].price

    mid_A = (best_bid_A + best_ask_A) / 2
    mid_B = (best_bid_B + best_ask_B) / 2

    # One unified signal avoids contradictory long/short target intervals.
    S_mid = (mid_A - mid_B) / S_MAX

    spread_position = calculate_spread_position(positions[stock_A], positions[stock_B])
    P = spread_position / PAIR_POSITION_LIMIT

    first_target, second_target = stellage_targets(S_mid)
    lower_target = min(first_target, second_target)
    upper_target = max(first_target, second_target)

    buy_pair_cost = best_ask_A - best_bid_B
    sell_pair_edge = best_bid_A - best_ask_B

    print(
        'Signal: S_mid=%.3f | P=%.3f | target=[%.3f, %.3f] | '
        'buy_cost=%.2f | sell_edge=%.2f'
        % (S_mid, P, lower_target, upper_target, buy_pair_cost, sell_pair_edge)
    )

    if P < lower_target:
        target_position = lower_target * PAIR_POSITION_LIMIT
        desired_volume = min(MAX_PAIR_VOLUME, max(1, int(round(target_position - spread_position))))

        # Do not repeatedly pay an unfavourable pair crossing cost.
        if buy_pair_cost <= MAX_STELLAGE_CROSS_COST:
            stellage_attempts += 1
            paired = execute_pair('buy_spread', desired_volume, 'STELLAGE BUY')
            if paired > 0:
                stellage_trades += 1
                stellage_lots += paired
                return True
        else:
            print('Stellage buy target exists, but IOC pair cost is too high.')

    elif P > upper_target:
        target_position = upper_target * PAIR_POSITION_LIMIT
        desired_volume = min(MAX_PAIR_VOLUME, max(1, int(round(spread_position - target_position))))

        if sell_pair_edge >= -MAX_STELLAGE_CROSS_COST:
            stellage_attempts += 1
            paired = execute_pair('sell_spread', desired_volume, 'STELLAGE SELL')
            if paired > 0:
                stellage_trades += 1
                stellage_lots += paired
                return True
        else:
            print('Stellage sell target exists, but IOC pair cost is too high.')

    if sell_pair_edge >= CROSSED_MIN_EDGE:
        crossed_attempts += 1
        paired = execute_pair('sell_spread', MAX_PAIR_VOLUME, 'CROSSED SELL')
        if paired > 0:
            crossed_trades += 1
            crossed_lots += paired
            return True

    if buy_pair_cost <= -CROSSED_MIN_EDGE:
        crossed_attempts += 1
        paired = execute_pair('buy_spread', MAX_PAIR_VOLUME, 'CROSSED BUY')
        if paired > 0:
            crossed_trades += 1
            crossed_lots += paired
            return True

    print('No safe active opportunity detected.')
    return False


# Spill balancing
# auto sent IOC to balance one-sided risks.
def position_balancer():
    global balance_attempts, balance_trades, balance_lots

    positions_before = exchange.get_positions()
    pos_A = positions_before[stock_A]
    pos_B = positions_before[stock_B]
    spill = calculate_spill(pos_A, pos_B)

    if abs(spill) <= SPILL_TOLERANCE:
        print(f'Spill {spill} is within tolerance +/-{SPILL_TOLERANCE}.')
        return False

    order_book_A = exchange.get_last_price_book(stock_A)
    order_book_B = exchange.get_last_price_book(stock_B)

    if spill > 0:
        #sell the instrument with the better best bid.
        choices = []
        if order_book_A.bids:
            choices.append((order_book_A.bids[0].price, stock_A, order_book_A.bids[0].volume))
        if order_book_B.bids:
            choices.append((order_book_B.bids[0].price, stock_B, order_book_B.bids[0].volume))
        if not choices:
            return False

        price, instrument, top_volume = max(choices, key=lambda item: item[0])
        amount = min(int(spill), MAX_BALANCE_VOLUME, int(top_volume), available_sell_volume(instrument))
        side = 'ask'

    else:
        #buy the instrument with the cheaper best ask.
        choices = []
        if order_book_A.asks:
            choices.append((order_book_A.asks[0].price, stock_A, order_book_A.asks[0].volume))
        if order_book_B.asks:
            choices.append((order_book_B.asks[0].price, stock_B, order_book_B.asks[0].volume))
        if not choices:
            return False

        price, instrument, top_volume = min(choices, key=lambda item: item[0])
        amount = min(int(abs(spill)), MAX_BALANCE_VOLUME, int(top_volume), available_buy_volume(instrument))
        side = 'bid'

    if amount <= 0:
        print('Balancer has no safe volume available.')
        return False

    balance_attempts += 1
    submitted = send_order(instrument, price, amount, side, 'ioc')
    if submitted <= 0:
        return False

    positions_after = exchange.get_positions()
    actual_fill = filled_volume(
        positions_before[instrument],
        positions_after[instrument],
        side,
    )

    if actual_fill > 0:
        balance_trades += 1
        balance_lots += actual_fill
        print(
            f'BALANCE: {side} {actual_fill} {instrument} @ {price}; '
            f'old spill={spill}'
        )
        return True

    return False


# passive market making
def market_make():
    order_book_A = exchange.get_last_price_book(stock_A)
    order_book_B = exchange.get_last_price_book(stock_B)

    if (not order_book_A.bids or not order_book_A.asks or
            not order_book_B.bids or not order_book_B.asks):
        return

    mid_A = (order_book_A.bids[0].price + order_book_A.asks[0].price) / 2
    mid_B = (order_book_B.bids[0].price + order_book_B.asks[0].price) / 2

    # Both dual-listed products use one shared pair fair value. This reduces
    # the risk of quoting around a temporarily mispriced individual midpoint.
    common_fair = (mid_A + mid_B) / 2

    positions = exchange.get_positions()

    for instrument in (stock_A, stock_B):
        inventory = positions[instrument]
        skew = MAX_INVENTORY_SKEW * (inventory / POSITION_LIMIT)

        bid_price = common_fair - PASSIVE_HALF_SPREAD - skew
        ask_price = common_fair + PASSIVE_HALF_SPREAD - skew

        buy_volume = min(PASSIVE_VOLUME, available_buy_volume(instrument))
        sell_volume = min(PASSIVE_VOLUME, available_sell_volume(instrument))

        # Do not submit quotes that cross the current visible market.
        book = order_book_A if instrument == stock_A else order_book_B

        if buy_volume > 0 and bid_price < book.asks[0].price:
            send_order(instrument, bid_price, buy_volume, 'bid', 'limit')

        if sell_volume > 0 and ask_price > book.bids[0].price:
            send_order(instrument, ask_price, sell_volume, 'ask', 'limit')


# reporting
def print_positions():
    positions = exchange.get_positions()
    pos_A = positions[stock_A]
    pos_B = positions[stock_B]
    spread = calculate_spread_position(pos_A, pos_B)
    spill = calculate_spill(pos_A, pos_B)

    print(
        f'Positions: {stock_A}={pos_A:+d}, {stock_B}={pos_B:+d}, '
        f'spread={spread:+d}, spill={spill:+d}'
    )


def success_rate(successes, attempts):
    if attempts == 0:
        return 0.0
    return 100.0 * successes / attempts


def print_final_report():
    elapsed = (datetime.datetime.now() - start_time).total_seconds()
    final_pnl = exchange.get_pnl()
    pnl_change = final_pnl - initial_pnl
    positions = exchange.get_positions()
    spread = calculate_spread_position(positions[stock_A], positions[stock_B])
    spill = calculate_spill(positions[stock_A], positions[stock_B])

    print()
    print('=' * 64)
    print('                    FINAL TRADING REPORT')
    print('=' * 64)
    print('Run duration:            %.0f seconds (%.1f minutes)' %
          (elapsed, elapsed / 60))
    print('Initial P&L:             $%.2f' % initial_pnl)
    print('Final P&L:               $%.2f' % final_pnl)
    print('P&L change:              $%.2f' % pnl_change)
    print(f'Final {stock_A} position:       {positions[stock_A]:+d}')
    print(f'Final {stock_B} position:  {positions[stock_B]:+d}')
    print(f'Final paired spread:      {spread:+d}')
    print(f'Final spill:              {spill:+d}')
    print('-' * 64)
    print(f'Total orders sent:        {total_orders_sent}')
    print(f'Active IOC orders sent:   {active_orders_sent}')
    print(f'Passive limit orders sent:{passive_orders_sent}')
    total_successes = stellage_trades + crossed_trades + balance_trades
    total_attempts = stellage_attempts + crossed_attempts + balance_attempts

    print(
        'Stellage success/attempts/lots: '
        f'{stellage_trades} / {stellage_attempts} / {stellage_lots} '
        f'({success_rate(stellage_trades, stellage_attempts):.1f}%)'
    )
    print(
        'Crossed success/attempts/lots:  '
        f'{crossed_trades} / {crossed_attempts} / {crossed_lots} '
        f'({success_rate(crossed_trades, crossed_attempts):.1f}%)'
    )
    print(
        'Balance success/attempts/lots:  '
        f'{balance_trades} / {balance_attempts} / {balance_lots} '
        f'({success_rate(balance_trades, balance_attempts):.1f}%)'
    )
    print(
        'Total active successes:         '
        f'{total_successes} / {total_attempts} '
        f'({success_rate(total_successes, total_attempts):.1f}%)'
    )
    print(f'Rate-limit blocks:        {rate_limit_blocks}')
    print('=' * 64)


# Main program
exchange = Exchange()
exchange.connect()

start_time = datetime.datetime.now()
initial_pnl = exchange.get_pnl()

print('Trading started at:', start_time)
print('Will run for %d seconds (%.1f minutes).' %
      (RUN_DURATION, RUN_DURATION / 60))
print('Initial P&L: %.2f' % initial_pnl)
print_positions()

last_status_second = -1

try:
    while True:
        elapsed = (datetime.datetime.now() - start_time).total_seconds()
        if elapsed >= RUN_DURATION:
            break

        # print a compact status line every 10 seconds
        status_second = int(elapsed) // 10
        if status_second != last_status_second:
            last_status_second = status_second
            print()
            print('Elapsed: %.1f / %d seconds | P&L change: %.2f' %
                  (elapsed, RUN_DURATION, exchange.get_pnl() - initial_pnl))
            print_positions()

        quotes_cleared = cancel_all_quotes()

        active_trade_this_loop = False
        if quotes_cleared:
            active_trade_this_loop = trade_loop()

        position_balancer()

        if not active_trade_this_loop:
            market_make()

        time.sleep(LOOP_SLEEP)

finally:
    try:
        exchange.delete_orders(stock_A)
        exchange.delete_orders(stock_B)
    except Exception as error:
        print('Final order cancellation failed:', error)

    print_final_report()